# 05 — Hard Negative Mining

This notebook converts the **errors and confusions of the baseline embedding model**
into stronger training examples.

Notebook 04 asked:

> How well does the original BGE-M3 retrieve Portuguese legal passages?

This notebook asks:

> Which *wrong* passages does BGE-M3 consider most similar to each training query?

Those passages are called **hard negatives**.

## Why negatives matter

An embedding model learns useful retrieval boundaries when it sees both:

```text
Query
  ├── Positive → passage that should be close
  └── Negative → passage that should be farther away
```

A random negative can be too easy:

```text
Query: "localização das operações para efeitos de IVA"

Positive: CIVA — rules on location of transactions
Random negative: unrelated provision on income tax
```

The model may already distinguish these perfectly.

A hard negative is more informative:

```text
Query: "localização das operações para efeitos de IVA"

Positive: correct CIVA article
Hard negative: another VAT article using highly similar terminology
```

The baseline model already places that wrong passage near the query. Fine-tuning can
therefore focus on distinctions the model currently finds difficult.

## Where this notebook fits

```text
03 — Retrieval dataset
        ↓
04 — Baseline BGE-M3
        ↓
baseline similarity rankings
        ↓
05 — Hard-negative mining       ← this notebook
        ↓
(query, positive, hard negative)
        ↓
06 — Fine-tuning
```

## Data-leakage rule

Hard negatives are mined **only from training articles**.

Validation and test articles must not influence model optimisation, even indirectly
through negative selection.

## Learning objectives

After this notebook, you should understand:

- the difference between random, structural and hard negatives;
- how dense retrieval can be reused to mine training examples;
- why the baseline model itself is used to identify difficult negatives;
- why relevant passages and sibling chunks must be excluded;
- why validation/test passages must not be used for mining;
- how similarity and baseline rank describe negative difficulty;
- how mined negatives are transformed into fine-tuning triplets.

## 1. Install dependencies if necessary

The mining stage uses the same embedding stack as baseline evaluation:
Sentence Transformers, PyTorch, NumPy and pandas.

If the environment was already prepared for notebook 04, no additional installation
should normally be required.

Keeping the same library environment also reduces the risk that baseline and mining
embeddings are produced by different software versions.

In [1]:
# Uncomment only if required.
# %pip install -U sentence-transformers torch pandas numpy tqdm

## 2. Imports and project paths

Inputs come from `data/dataset_v2/`:

```text
passages.jsonl
train_queries.jsonl
train_qrels.jsonl
```

Only **training queries and training relevance judgements** are loaded.

Outputs are separated into:

```text
data/hard_negatives/
    machine-learning records

results/hard_negative_mining/
    diagnostics and experiment reports
```

This keeps generated training data distinct from analytical results.

In [2]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import json
import time

import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

PROJECT_ROOT = Path("..")

DATASET_DIR = PROJECT_ROOT / "data" / "dataset_v2"
OUTPUT_DIR = PROJECT_ROOT / "data" / "hard_negatives"
RESULTS_DIR = PROJECT_ROOT / "results" / "hard_negative_mining"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PASSAGES_PATH = DATASET_DIR / "passages.jsonl"
TRAIN_QUERIES_PATH = DATASET_DIR / "train_queries.jsonl"
TRAIN_QRELS_PATH = DATASET_DIR / "train_qrels.jsonl"

print(f"Dataset : {DATASET_DIR.resolve()}")
print(f"Output  : {OUTPUT_DIR.resolve()}")
print(f"Results : {RESULTS_DIR.resolve()}")

C:\Users\user\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset : C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\data\dataset_v2
Output  : C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\data\hard_negatives
Results : C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\results\hard_negative_mining


## 3. Mining configuration

The same original `BAAI/bge-m3` model used for the baseline is used for mining.

### `MINING_TOP_K = 50`

For each query, we first inspect the 50 passages that the baseline model considers most
similar.

We do **not** automatically use all 50. Relevant passages, sibling chunks and
ineligible articles are removed first.

### `HARD_NEGATIVES_PER_QUERY = 5`

After filtering, at most five high-ranking non-relevant passages are retained.

This creates several difficult alternatives for each query without allowing one query
to dominate the training dataset.

### `MIN_SIMILARITY = None`

No arbitrary similarity cut-off is imposed in this version. Similarity and original
baseline rank are stored so difficulty can be analysed later.

### Why keep mining conservative?

A passage with extremely high similarity may be a useful hard negative — but it may
also expose an incomplete qrel or an ambiguous query. Manual inspection therefore
remains part of the workflow.

In [3]:
MODEL_NAME = "BAAI/bge-m3"

BATCH_SIZE = 32
MAX_SEQ_LENGTH = 512

MINING_TOP_K = 50
HARD_NEGATIVES_PER_QUERY = 5

# The first non-relevant result can sometimes be almost indistinguishable from
# the positive. We retain rank and score information rather than imposing an
# arbitrary similarity threshold at this stage.
MIN_SIMILARITY = None

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(f"Model  : {MODEL_NAME}")
print(f"Device : {DEVICE}")

if DEVICE == "cuda":
    print(
        "GPU    :",
        torch.cuda.get_device_name(0)
    )
else:
    print(
        "WARNING: CUDA is not available. "
        "Mining will run on CPU and may be slow."
    )

Model  : BAAI/bge-m3
Device : cpu


## 4. JSONL loader

This is the same basic loading pattern used in the previous notebooks.

Each non-empty line is parsed independently and malformed JSON produces an explicit
error containing the line number.

Reusing simple, deterministic I/O helpers helps keep attention on the experimental
logic rather than file-handling differences.

In [4]:
def read_jsonl(path):
    records = []

    with path.open(
        "r",
        encoding="utf-8"
    ) as f:
        for line_number, line in enumerate(
            f,
            start=1
        ):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(
                    json.loads(line)
                )
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid JSON in {path.name}, "
                    f"line {line_number}: {exc}"
                )

    return records

## 5. Load training data

Hard-negative mining is a **training-data construction operation**, so we load:

- the complete passage corpus;
- training queries;
- training qrels.

The complete passage file is needed initially because it contains all passage metadata,
but later we explicitly restrict the eligible negative pool to articles assigned to
training.

The query-type count also lets us verify that both semantic and reference queries are
represented.

In [5]:
passages = read_jsonl(
    PASSAGES_PATH
)

train_queries = read_jsonl(
    TRAIN_QUERIES_PATH
)

train_qrels = read_jsonl(
    TRAIN_QRELS_PATH
)

print(f"Passages      : {len(passages)}")
print(f"Train queries : {len(train_queries)}")
print(f"Train qrels   : {len(train_qrels)}")

print(
    "Query types   :",
    Counter(
        query["query_type"]
        for query in train_queries
    )
)

Passages      : 2088
Train queries : 4224
Train qrels   : 6752
Query types   : Counter({'semantic': 3165, 'reference': 1059})


## 6. Validate IDs and relevance mappings

Before mining, we reconstruct the ground-truth relationship:

```text
query_id → {relevant passage IDs}
```

and verify:

- passage IDs are unique;
- query IDs are unique;
- every training query has a relevant passage;
- every relevant passage actually exists.

Hard-negative mining depends critically on correct qrels. If a truly relevant passage
is missing from the qrels, the algorithm could mistakenly label it as a negative.

That would create a **false negative**, one of the most damaging forms of label noise
for contrastive retrieval training.

In [6]:
passage_by_id = {
    passage["passage_id"]: passage
    for passage in passages
}

assert len(passage_by_id) == len(
    passages
), "Duplicate passage IDs detected."


query_by_id = {
    query["query_id"]: query
    for query in train_queries
}

assert len(query_by_id) == len(
    train_queries
), "Duplicate query IDs detected."


relevant_by_query = defaultdict(set)

for qrel in train_qrels:
    if qrel.get("relevance", 0) > 0:
        relevant_by_query[
            qrel["query_id"]
        ].add(
            qrel["passage_id"]
        )


errors = []

for query in train_queries:
    query_id = query["query_id"]

    if not relevant_by_query[query_id]:
        errors.append(
            f"No relevant passage: {query_id}"
        )

    for passage_id in relevant_by_query[
        query_id
    ]:
        if passage_id not in passage_by_id:
            errors.append(
                f"Unknown passage: {passage_id}"
            )


print(f"Validation errors: {len(errors)}")

for error in errors[:50]:
    print(error)

assert not errors

Validation errors: 0


## 7. Load the baseline embedding model

The original, unfine-tuned BGE-M3 model is loaded again.

This is intentional: hard negatives should represent the **baseline model's current
confusions**.

If we mined with a different model, the negatives would reflect that other model's
embedding space rather than the weaknesses measured in notebook 04.

No model parameters are modified in this notebook.

In [7]:
start = time.time()

model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE
)

model.max_seq_length = MAX_SEQ_LENGTH

print(
    f"Loaded in "
    f"{time.time() - start:.1f} s"
)

print(
    "Embedding dimension:",
    model.get_sentence_embedding_dimension()
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 17712.01it/s]


Loaded in 7.0 s
Embedding dimension: 1024


C:\Users\user\AppData\Local\Temp\ipykernel_11044\1644774456.py:17: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()


## 8. Encode all passages

As in baseline evaluation, every passage is converted into a normalised embedding.

Normalisation means similarity can later be calculated using the dot product.

Although all passages are encoded here, not all of them are eligible as negatives.
The next stage applies the article-level training boundary.

This separation is useful because passage embeddings retain the same ordering as the
master passage corpus.

In [8]:
passage_texts = [
    passage["text"]
    for passage in passages
]

start = time.time()

passage_embeddings = model.encode(
    passage_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

passage_embeddings = np.asarray(
    passage_embeddings,
    dtype=np.float32
)

print(
    "Passage embedding matrix:",
    passage_embeddings.shape
)

print(
    f"Encoding time: "
    f"{time.time() - start:.1f} s"
)

Batches: 100%|██████████| 66/66 [31:51<00:00, 28.97s/it]

Passage embedding matrix: (2088, 1024)
Encoding time: 1912.0 s


## 9. Restrict negative mining to training articles

This is the central anti-leakage safeguard.

The dataset split was created at article level in notebook 03. We collect the article
IDs belonging to training queries and allow negatives only from those articles.

Therefore:

```text
TRAIN query
   ↓
candidate negatives
   ├── TRAIN article       ✓ allowed
   ├── VALIDATION article  ✗ excluded
   └── TEST article        ✗ excluded
```

Why does this matter?

If a test passage were selected as a hard negative, its text would later appear in a
fine-tuning triplet. The model would then have been trained using information from the
test set, invalidating the final evaluation.

Data leakage can occur through negatives just as easily as through positives.

In [9]:
train_article_ids = {
    query["source_article_id"]
    for query in train_queries
}

eligible_passage_indices = [
    index
    for index, passage in enumerate(
        passages
    )
    if passage["source_article_id"]
    in train_article_ids
]

eligible_passages = [
    passages[index]
    for index in eligible_passage_indices
]

eligible_embeddings = (
    passage_embeddings[
        eligible_passage_indices
    ]
)

print(
    f"Eligible training passages: "
    f"{len(eligible_passages)}"
)

print(
    f"Excluded validation/test passages: "
    f"{len(passages) - len(eligible_passages)}"
)

# Notice that eligibility is defined through the article split, not merely by
# passage IDs. This preserves the article-level leakage boundary established in
# notebook 03.


Eligible training passages: 1691
Excluded validation/test passages: 397


## 10. Encode training queries

Training queries are encoded with the same baseline model and the same normalisation
strategy as passages.

We now have:

```text
query embeddings     → Q
eligible passage embeddings → P
```

For each query, hard-negative mining will compare its vector with every eligible
training passage vector.

In [10]:
query_texts = [
    query["query"]
    for query in train_queries
]

start = time.time()

query_embeddings = model.encode(
    query_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

query_embeddings = np.asarray(
    query_embeddings,
    dtype=np.float32
)

print(
    "Query embedding matrix:",
    query_embeddings.shape
)

print(
    f"Encoding time: "
    f"{time.time() - start:.1f} s"
)

Batches: 100%|██████████| 132/132 [04:08<00:00,  1.89s/it]

Query embedding matrix: (4224, 1024)
Encoding time: 249.0 s


## 11. Mine nearest non-relevant passages

For each training query, the algorithm performs dense retrieval and then filters the
ranking.

### Step 1 — Similarity

```text
scores = eligible_passage_embeddings × query_embedding
```

Because vectors are normalised, these are cosine-similarity scores.

### Step 2 — Retrieve baseline candidates

The top `MINING_TOP_K` candidates are selected and ordered from highest to lowest
similarity.

### Step 3 — Remove known relevant passages

Any passage explicitly marked relevant in the qrels is excluded.

### Step 4 — Remove sibling chunks

This stronger rule excludes **every passage from the query's source article**, even if
a particular chunk is not explicitly listed as the chosen positive.

Without this rule:

```text
Article 10 — chunk 1 → positive
Article 10 — chunk 2 → negative   ✗ contradiction
```

### Step 5 — Retain difficult negatives

The first remaining high-scoring passages become hard negatives.

For each one we preserve:

- similarity score;
- original baseline rank;
- source document and article;
- whether it comes from the same legal document.

The stored `baseline_rank` is the rank **before filtering**, which tells us how highly
the original model ranked that wrong passage.

In [11]:
def mine_query_negatives(
    query,
    query_embedding,
    top_k=MINING_TOP_K,
    negatives_per_query=HARD_NEGATIVES_PER_QUERY
):
    scores = (
        eligible_embeddings
        @ query_embedding
    )

    candidate_count = min(
        top_k,
        len(eligible_passages)
    )

    top_indices = np.argpartition(
        -scores,
        candidate_count - 1
    )[:candidate_count]

    top_indices = top_indices[
        np.argsort(
            -scores[top_indices]
        )
    ]

    relevant_ids = relevant_by_query[
        query["query_id"]
    ]

    negatives = []

    for baseline_rank, local_index in enumerate(
        top_indices,
        start=1
    ):
        passage = eligible_passages[
            local_index
        ]

        passage_id = passage[
            "passage_id"
        ]

        score = float(
            scores[local_index]
        )

        # Explicit qrel exclusion.
        if passage_id in relevant_ids:
            continue

        # Stronger article-level exclusion:
        # no sibling chunk may be a negative.
        if (
            passage["source_article_id"]
            == query["source_article_id"]
        ):
            continue

        if (
            MIN_SIMILARITY is not None
            and score < MIN_SIMILARITY
        ):
            continue

        negatives.append({
            "passage_id": passage_id,
            "source_article_id": (
                passage["source_article_id"]
            ),
            "document_id": (
                passage["document_id"]
            ),
            "article": (
                passage["article"]
            ),
            "article_title": (
                passage.get("article_title")
            ),
            "text": passage["text"],
            "similarity": score,
            "baseline_rank": baseline_rank,
            "same_document": (
                passage["document_id"]
                == query["document_id"]
            ),
        })

        if (
            len(negatives)
            >= negatives_per_query
        ):
            break

    return negatives

# The filtering order is deliberate: retrieve what the baseline considers
# difficult first, then remove anything that cannot legally serve as a negative.


In [12]:
mined_records = []

for index, query in enumerate(
    tqdm(
        train_queries,
        desc="Mining hard negatives"
    )
):
    negatives = mine_query_negatives(
        query,
        query_embeddings[index]
    )

    mined_records.append({
        "query_id": query["query_id"],
        "query_type": query["query_type"],
        "query": query["query"],
        "document_id": query["document_id"],
        "article": query["article"],
        "article_title": (
            query.get("article_title")
        ),
        "source_article_id": (
            query["source_article_id"]
        ),
        "relevant_passage_ids": sorted(
            relevant_by_query[
                query["query_id"]
            ]
        ),
        "hard_negatives": negatives,
        "hard_negative_count": len(
            negatives
        ),
    })


print(
    "Queries mined:",
    len(mined_records)
)

print(
    "Negative counts:",
    Counter(
        record["hard_negative_count"]
        for record in mined_records
    )
)

Mining hard negatives: 100%|██████████| 4224/4224 [00:01<00:00, 3643.68it/s]

Queries mined: 4224
Negative counts: Counter({5: 4224})


## 12. Mining quality checks

Before trusting the mined examples, we verify that:

- the same negative is not duplicated for one query;
- no qrel-relevant passage is used as a negative;
- no sibling chunk from the target article is used as a negative;
- no validation/test article enters the negative pool.

These assertions formalise the intended experimental rules.

If one fails, the correct response is to fix the mining logic or dataset — not to
ignore the error and continue to fine-tuning.

In [13]:
quality_errors = []

for record in mined_records:
    relevant_ids = set(
        record["relevant_passage_ids"]
    )

    negative_ids = [
        negative["passage_id"]
        for negative in record[
            "hard_negatives"
        ]
    ]

    if len(negative_ids) != len(
        set(negative_ids)
    ):
        quality_errors.append({
            "query_id": record["query_id"],
            "error": "duplicate_negative"
        })

    for negative in record[
        "hard_negatives"
    ]:
        if (
            negative["passage_id"]
            in relevant_ids
        ):
            quality_errors.append({
                "query_id": record["query_id"],
                "error": "relevant_used_as_negative",
                "passage_id": (
                    negative["passage_id"]
                )
            })

        if (
            negative["source_article_id"]
            == record["source_article_id"]
        ):
            quality_errors.append({
                "query_id": record["query_id"],
                "error": "sibling_chunk_used_as_negative",
                "passage_id": (
                    negative["passage_id"]
                )
            })

        if (
            negative["source_article_id"]
            not in train_article_ids
        ):
            quality_errors.append({
                "query_id": record["query_id"],
                "error": "evaluation_article_used_as_negative",
                "passage_id": (
                    negative["passage_id"]
                )
            })


print(
    f"Quality errors: "
    f"{len(quality_errors)}"
)

for error in quality_errors[:50]:
    print(error)

assert not quality_errors

Quality errors: 0


## 13. Mining statistics

We now describe the difficulty and composition of the mined negatives.

Useful indicators include:

### Mean similarity

Higher similarity means the baseline embedding model places the negative closer to the
query.

### Mean baseline rank

A negative originally appearing at rank 2 is generally a stronger baseline confusion
than one appearing at rank 45.

### Same-document ratio

This measures how often the hard negative comes from the same legal document as the
target.

Same-document negatives can be especially useful because they often share vocabulary,
tax concepts and drafting style with the positive.

These statistics describe the mining process; they are not model-quality metrics like
Recall or MRR.

In [14]:
all_negatives = [
    negative
    for record in mined_records
    for negative in record[
        "hard_negatives"
    ]
]

similarities = [
    negative["similarity"]
    for negative in all_negatives
]

ranks = [
    negative["baseline_rank"]
    for negative in all_negatives
]

same_document_count = sum(
    negative["same_document"]
    for negative in all_negatives
)

mining_statistics = {
    "queries": len(mined_records),
    "queries_with_negatives": sum(
        record["hard_negative_count"] > 0
        for record in mined_records
    ),
    "hard_negatives": len(
        all_negatives
    ),
    "mean_negatives_per_query": (
        len(all_negatives)
        / len(mined_records)
        if mined_records
        else 0.0
    ),
    "mean_similarity": (
        float(np.mean(similarities))
        if similarities
        else None
    ),
    "median_similarity": (
        float(np.median(similarities))
        if similarities
        else None
    ),
    "min_similarity": (
        float(np.min(similarities))
        if similarities
        else None
    ),
    "max_similarity": (
        float(np.max(similarities))
        if similarities
        else None
    ),
    "mean_baseline_rank": (
        float(np.mean(ranks))
        if ranks
        else None
    ),
    "same_document_negatives": (
        same_document_count
    ),
    "same_document_ratio": (
        same_document_count
        / len(all_negatives)
        if all_negatives
        else 0.0
    ),
}

mining_statistics

{'queries': 4224,
 'queries_with_negatives': 4224,
 'hard_negatives': 21120,
 'mean_negatives_per_query': 5.0,
 'mean_similarity': 0.5834062809322142,
 'median_similarity': 0.5838114023208618,
 'min_similarity': 0.32595500349998474,
 'max_similarity': 0.7771579027175903,
 'mean_baseline_rank': 4.136979166666666,
 'same_document_negatives': 16663,
 'same_document_ratio': 0.788967803030303}

## 14. Statistics by query type

We compare hard negatives mined for semantic and reference queries.

Reference queries contain explicit identifiers and may therefore create a different
similarity landscape from semantic questions.

For each type we inspect:

- number of queries;
- number of negatives;
- mean negative similarity;
- proportion from the same document;
- mean baseline rank.

Differences here can help explain later differences in fine-tuning behaviour.

In [15]:
type_rows = []

for query_type in [
    "semantic",
    "reference"
]:
    records = [
        record
        for record in mined_records
        if record["query_type"]
        == query_type
    ]

    negatives = [
        negative
        for record in records
        for negative in record[
            "hard_negatives"
        ]
    ]

    type_rows.append({
        "query_type": query_type,
        "queries": len(records),
        "negatives": len(negatives),
        "mean_similarity": (
            np.mean([
                n["similarity"]
                for n in negatives
            ])
            if negatives
            else np.nan
        ),
        "same_document_ratio": (
            np.mean([
                n["same_document"]
                for n in negatives
            ])
            if negatives
            else np.nan
        ),
        "mean_baseline_rank": (
            np.mean([
                n["baseline_rank"]
                for n in negatives
            ])
            if negatives
            else np.nan
        ),
    })


type_stats_df = pd.DataFrame(
    type_rows
)

type_stats_df

,query_type,queries,negatives,mean_similarity,same_document_ratio,mean_baseline_rank
0,semantic,3165,15825,0.595147,0.734976,4.368215
1,reference,1059,5295,0.548318,0.950331,3.445892


## 15. Statistics by legal document

Mining is also analysed separately for each source document.

A high same-document ratio may indicate that the model frequently confuses nearby
provisions inside that code.

A lower ratio may indicate that conceptually similar provisions across different laws
are being retrieved.

Neither is automatically good or bad; the purpose is to understand what distinctions
the future fine-tuning dataset is asking the model to learn.

In [16]:
document_rows = []

documents = sorted(
    set(
        record["document_id"]
        for record in mined_records
    )
)

for document_id in documents:
    records = [
        record
        for record in mined_records
        if record["document_id"]
        == document_id
    ]

    negatives = [
        negative
        for record in records
        for negative in record[
            "hard_negatives"
        ]
    ]

    document_rows.append({
        "document_id": document_id,
        "queries": len(records),
        "negatives": len(negatives),
        "mean_similarity": (
            np.mean([
                n["similarity"]
                for n in negatives
            ])
            if negatives
            else np.nan
        ),
        "same_document_ratio": (
            np.mean([
                n["same_document"]
                for n in negatives
            ])
            if negatives
            else np.nan
        ),
        "mean_baseline_rank": (
            np.mean([
                n["baseline_rank"]
                for n in negatives
            ])
            if negatives
            else np.nan
        ),
    })


document_stats_df = pd.DataFrame(
    document_rows
)

document_stats_df

,document_id,queries,negatives,mean_similarity,same_document_ratio,mean_baseline_rank
0,cimi,488,2440,0.559810,0.821311,3.814344
1,circ,532,2660,0.591897,0.757519,4.374436
2,cirs,608,3040,0.594480,0.669737,4.175987
3,cis,232,1160,0.600064,0.757759,4.059483
4,civa,320,1600,0.615297,0.821250,4.453125
5,ebf,385,1925,0.607343,0.834805,4.517922
6,lgt,373,1865,0.588160,0.816086,4.155496
7,oe2026,793,3965,0.572064,0.884489,3.967970
8,rgit,389,1945,0.547295,0.801542,3.923907
9,riti,104,520,0.566563,0.423077,4.017308


## 16. Inspect representative semantic hard negatives

Statistics alone cannot tell us whether a negative is *actually* non-relevant.

We therefore inspect semantic queries together with their highest-ranked negatives.

For each example, ask:

- Is the negative genuinely wrong?
- Is it nevertheless semantically plausible?
- Does it belong to the same legal topic?
- Could the query reasonably have more than one relevant article?
- Is the qrel incomplete?

The ideal hard negative is:

> **plausible enough to confuse the model, but clearly not a correct answer to the
> query under the dataset's relevance definition.**

In [17]:
semantic_records = [
    record
    for record in mined_records
    if (
        record["query_type"]
        == "semantic"
        and record[
            "hard_negative_count"
        ] > 0
    )
]

sample_count = min(
    15,
    len(semantic_records)
)

sample_indices = np.linspace(
    0,
    len(semantic_records) - 1,
    sample_count,
    dtype=int
) if sample_count else []


for index in sample_indices:
    record = semantic_records[
        index
    ]

    print("\n" + "=" * 100)
    print(
        f"{record['document_id'].upper()} | "
        f"Art. {record['article']}"
    )

    print("\nQUERY")
    print(record["query"])

    print("\nTARGET ARTICLE")
    print(
        record["source_article_id"]
    )

    print("\nTOP HARD NEGATIVES")

    for rank, negative in enumerate(
        record["hard_negatives"][:3],
        start=1
    ):
        print(
            f"\n[{rank}] "
            f"similarity={negative['similarity']:.4f} | "
            f"baseline_rank={negative['baseline_rank']} | "
            f"same_document={negative['same_document']}"
        )

        print(
            negative["text"][:1000]
        )


CIMI | Art. 1.º

QUERY
Quais são as regras sobre Incidência?

TARGET ARTICLE
cimi_block_02_art_1_o

TOP HARD NEGATIVES

[1] similarity=0.5387 | baseline_rank=1 | same_document=False
Documento: CIRS
Estrutura: Section II — Incidência pessoal
Artigo 16.º — Residência
3 - O disposto nos n.os 8 a 12 do artigo 16.o, no artigo 22.o, nos n.os 10 e 12 do artigo 72.o, dos n.os 4 a 8 do artigo 81.o, no n.o 8 do
artigo 99.o e na alínea d) do n.o 1 do artigo 101.o do Código do IRS, na redação anterior à introduzida pela presente lei, continua a
ser aplicável, até ao termo do prazo previsto no n.o 9 do artigo 16.o do Código do IRS, na redação anterior à introduzida pela presente
lei, contado da data em que o sujeito passivo se tornou residente em território português, ao sujeito passivo que:
a) À data da entrada em vigor da presente lei, já se encontre inscrito como residente não habitual no registo de contribuintes da AT,
enquanto não estiver esgotado o período a que se referem os n.os 9 a 12 do 

## 17. Inspect the hardest negatives

This view sorts all mined negatives by similarity.

The highest-similarity cases deserve particular attention because two very different
situations can produce the same numerical pattern:

```text
High similarity + genuinely wrong
→ excellent hard negative

High similarity + actually relevant
→ likely qrel/query problem
```

Hard-negative mining therefore doubles as a **dataset-auditing tool**. The model can
reveal semantic relationships that simple article-derived qrels did not capture.

In [18]:
hardest = []

for record in mined_records:
    for negative in record[
        "hard_negatives"
    ]:
        hardest.append({
            "query_id": record["query_id"],
            "query_type": record["query_type"],
            "query": record["query"],
            "target_document": (
                record["document_id"]
            ),
            "target_article": (
                record["article"]
            ),
            **negative,
        })


hardest = sorted(
    hardest,
    key=lambda item: item["similarity"],
    reverse=True
)


for item in hardest[:30]:
    print("\n" + "-" * 100)

    print(
        f"Similarity: "
        f"{item['similarity']:.4f}"
    )

    print(
        f"Query: "
        f"{item['query']}"
    )

    print(
        f"Target: "
        f"{item['target_document'].upper()} "
        f"Art. {item['target_article']}"
    )

    print(
        f"Negative: "
        f"{item['document_id'].upper()} "
        f"Art. {item['article']} | "
        f"{item['passage_id']}"
    )


----------------------------------------------------------------------------------------------------
Similarity: 0.7772
Query: O que prevê o Estatuto dos Benefícios Fiscais relativamente a Regime aplicável às entidades licenciadas na Zona Franca da Madeira a partir de 1 de?
Target: EBF Art. 36.º-A
Negative: EBF Art. 36.º | ebf_block_05_art_36_o__chunk_01

----------------------------------------------------------------------------------------------------
Similarity: 0.7763
Query: O que prevê o Estatuto dos Benefícios Fiscais relativamente a Regime aplicável às entidades licenciadas na Zona Franca da Madeira a partir de 1 de?
Target: EBF Art. 36.º-A
Negative: EBF Art. 36.º | ebf_block_05_art_36_o__chunk_06

----------------------------------------------------------------------------------------------------
Similarity: 0.7710
Query: O que prevê o CIVA relativamente a Regularizações das deduções relativas a bens do activo imobilizado?
Target: CIVA Art. 24.º
Negative: CIVA Art. 25.º | civ

## 18. Build fine-tuning triplets

The mined records are now transformed into the format required by triplet-based
training:

```text
(anchor, positive, negative)

anchor   = query text
positive = relevant passage
negative = mined hard-negative passage
```

If a query has five hard negatives, it creates five triplets.

### Positive selection

This version deterministically selects the first sorted qrel-relevant passage as the
positive.

For multi-chunk articles, this is a simplifying design decision. Future experiments
could rotate positive chunks or use an objective that naturally supports multiple
positives.

### Stable triplet IDs

A SHA-256 hash of query, positive and negative IDs creates a deterministic triplet
identifier.

The same inputs therefore generate the same technical ID across reruns.

In [19]:
finetuning_triplets = []

for record in mined_records:
    relevant_ids = (
        record[
            "relevant_passage_ids"
        ]
    )

    if not relevant_ids:
        continue

    # Deterministic positive selection.
    positive_passage_id = (
        sorted(relevant_ids)[0]
    )

    positive = passage_by_id[
        positive_passage_id
    ]

    for negative_rank, negative in enumerate(
        record["hard_negatives"],
        start=1
    ):
        triplet_key = (
            record["query_id"]
            + "|"
            + positive_passage_id
            + "|"
            + negative["passage_id"]
        )

        triplet_id = hashlib.sha256(
            triplet_key.encode("utf-8")
        ).hexdigest()[:16]

        finetuning_triplets.append({
            "triplet_id": triplet_id,
            "query_id": (
                record["query_id"]
            ),
            "query_type": (
                record["query_type"]
            ),
            "query": record["query"],
            "positive": (
                positive["text"]
            ),
            "negative": (
                negative["text"]
            ),
            "positive_passage_id": (
                positive_passage_id
            ),
            "negative_passage_id": (
                negative["passage_id"]
            ),
            "source_article_id": (
                record[
                    "source_article_id"
                ]
            ),
            "negative_source_article_id": (
                negative[
                    "source_article_id"
                ]
            ),
            "document_id": (
                record["document_id"]
            ),
            "article": (
                record["article"]
            ),
            "negative_document_id": (
                negative["document_id"]
            ),
            "negative_article": (
                negative["article"]
            ),
            "negative_similarity": (
                negative["similarity"]
            ),
            "negative_baseline_rank": (
                negative[
                    "baseline_rank"
                ]
            ),
            "negative_same_document": (
                negative[
                    "same_document"
                ]
            ),
            "hard_negative_index": (
                negative_rank
            ),
        })


print(
    f"Fine-tuning triplets: "
    f"{len(finetuning_triplets)}"
)

print(
    "By query type:",
    Counter(
        t["query_type"]
        for t in finetuning_triplets
    )
)

# One query can generate several triplets because each hard negative provides
# a different contrastive learning signal against the same positive provision.


Fine-tuning triplets: 21120
By query type: Counter({'semantic': 15825, 'reference': 5295})


## 19. Final triplet validation

Before export, the generated training examples are checked again.

We verify that:

- triplet IDs are unique;
- positive and negative passage IDs differ;
- positive and negative do not originate from the same article.

This second validation layer is intentional. Mining records and final triplets are
different representations, so each should independently satisfy its own invariants.

In [20]:
triplet_errors = []

triplet_ids = [
    triplet["triplet_id"]
    for triplet in finetuning_triplets
]

if len(triplet_ids) != len(
    set(triplet_ids)
):
    triplet_errors.append(
        "duplicate_triplet_ids"
    )


for triplet in finetuning_triplets:
    if (
        triplet[
            "positive_passage_id"
        ]
        == triplet[
            "negative_passage_id"
        ]
    ):
        triplet_errors.append(
            (
                triplet["triplet_id"],
                "positive_equals_negative"
            )
        )

    if (
        triplet[
            "source_article_id"
        ]
        == triplet[
            "negative_source_article_id"
        ]
    ):
        triplet_errors.append(
            (
                triplet["triplet_id"],
                "same_article_negative"
            )
        )


print(
    f"Triplet errors: "
    f"{len(triplet_errors)}"
)

for error in triplet_errors[:50]:
    print(error)

assert not triplet_errors

Triplet errors: 0


## 20. Export mined negatives

`train_hard_negatives.jsonl` preserves the complete mining output per query.

This file is useful for inspection because one query record contains all of its mined
negatives together with scores, ranks and metadata.

It is the richer **analysis representation** of the mining process.

In [21]:
def write_jsonl(path, records):
    with path.open(
        "w",
        encoding="utf-8"
    ) as f:
        for record in records:
            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                )
                + "\n"
            )


MINED_PATH = (
    OUTPUT_DIR
    / "train_hard_negatives.jsonl"
)

write_jsonl(
    MINED_PATH,
    mined_records
)

print(MINED_PATH.resolve())

C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\data\hard_negatives\train_hard_negatives.jsonl


## 21. Export fine-tuning triplets

`train_hard_negative_triplets.jsonl` is the model-training representation.

Each line contains one:

```text
query + positive + hard negative
```

together with IDs and provenance metadata.

Notebook 06 can load this file directly without repeating the mining computation.

In [22]:
TRIPLETS_PATH = (
    OUTPUT_DIR
    / "train_hard_negative_triplets.jsonl"
)

write_jsonl(
    TRIPLETS_PATH,
    finetuning_triplets
)

print(TRIPLETS_PATH.resolve())

C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\data\hard_negatives\train_hard_negative_triplets.jsonl


## 22. Export mining statistics

The query-type and document-level statistics are saved as CSV.

These files provide compact evidence about how the hard-negative dataset was
constructed and can be used in experiment reports or visual analysis.

In [23]:
TYPE_STATS_PATH = (
    RESULTS_DIR
    / "hard_negative_by_query_type.csv"
)

DOCUMENT_STATS_PATH = (
    RESULTS_DIR
    / "hard_negative_by_document.csv"
)

type_stats_df.to_csv(
    TYPE_STATS_PATH,
    index=False
)

document_stats_df.to_csv(
    DOCUMENT_STATS_PATH,
    index=False
)

print(TYPE_STATS_PATH.resolve())
print(DOCUMENT_STATS_PATH.resolve())

C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\results\hard_negative_mining\hard_negative_by_query_type.csv
C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\results\hard_negative_mining\hard_negative_by_document.csv


## 23. Export mining report

The JSON report records both configuration and quality information:

- baseline model;
- device;
- embedding dimension;
- sequence length;
- mining top-K;
- negatives per query;
- similarity threshold;
- aggregate statistics;
- quality-check results.

This follows the same principle used in baseline evaluation:

> Results should be stored together with enough configuration to explain how they were
> produced.

In [24]:
REPORT_PATH = (
    RESULTS_DIR
    / "hard_negative_mining_report.json"
)

report = {
    "mining_version": "1.0",
    "model_name": MODEL_NAME,
    "device": DEVICE,
    "embedding_dimension": (
        model.get_sentence_embedding_dimension()
    ),
    "max_seq_length": (
        MAX_SEQ_LENGTH
    ),
    "batch_size": (
        BATCH_SIZE
    ),
    "mining_top_k": (
        MINING_TOP_K
    ),
    "hard_negatives_per_query": (
        HARD_NEGATIVES_PER_QUERY
    ),
    "min_similarity": (
        MIN_SIMILARITY
    ),
    "statistics": (
        mining_statistics
    ),
    "quality_errors": (
        quality_errors
    ),
    "triplet_errors": (
        triplet_errors
    ),
}


with REPORT_PATH.open(
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        report,
        f,
        ensure_ascii=False,
        indent=2
    )


print(REPORT_PATH.resolve())

# Persisting mining parameters is important because changing TOP_K or the
# number of negatives changes the training dataset even when the source corpus
# itself remains unchanged.


C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\results\hard_negative_mining\hard_negative_mining_report.json


C:\Users\user\AppData\Local\Temp\ipykernel_11044\639415150.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()


## 24. Final output structure

The notebook creates:

```text
data/
└── hard_negatives/
    ├── train_hard_negatives.jsonl
    └── train_hard_negative_triplets.jsonl

results/
└── hard_negative_mining/
    ├── hard_negative_by_query_type.csv
    ├── hard_negative_by_document.csv
    └── hard_negative_mining_report.json
```

The conceptual transformation is:

```text
Training queries
      +
Training passages
      +
Baseline BGE-M3
      ↓
Similarity ranking
      ↓
Remove positives
      ↓
Remove same-article chunks
      ↓
Remove validation/test articles
      ↓
Highest-ranked wrong passages
      ↓
HARD NEGATIVES
      ↓
Fine-tuning triplets
```

## What should be reviewed before training?

Pay particular attention to the highest-similarity semantic negatives.

Legal corpora contain overlapping terminology, cross-references and provisions that
regulate closely related concepts. A passage that appears to be a perfect hard
negative may actually reveal that the query has multiple legitimate answers.

Do not automatically interpret every mined record as unquestionable ground truth.

## 25. Next stage — fine-tuning

We now have the data required to adapt the embedding model:

```text
Query
  +
Relevant passage
  +
Hard negative
       ↓
06 — Fine-tuning
```

The purpose of training is to reshape the embedding space so that:

```text
similarity(query, positive)
            >
similarity(query, hard negative)
```

while preserving the general capabilities of the pretrained model as much as possible.

For a resource-constrained local experiment, the next notebook can use **LoRA/PEFT**:

```text
BGE-M3
568M+ parameters
      │
      ├── pretrained parameters → frozen
      │
      └── small LoRA adapters   → trainable
```

This reduces the number of trainable parameters and the memory required for gradients
and optimiser states.

The experimental discipline remains unchanged:

- train only on the training split;
- use validation for checkpoint/model selection;
- keep test untouched;
- record hyperparameters;
- compare against the frozen baseline from notebook 04.

The final question is not whether training loss decreases. It is:

> **Does the adapted model retrieve unseen Portuguese legal provisions better than
> the original BGE-M3?**

That can only be answered by repeating the retrieval evaluation on held-out data.